# ⚽ Project 1: Soccer Player Transfer Values
### FIFA World Cup 2026 — Reading the CSV and Exploring the Data

---

**Dataset:** `fifa_world_cup_2026_player_performance.csv`

**What you will learn:**
1. Load a real CSV into a pandas DataFrame
2. Inspect the DataFrame — shape, columns, types, missing values
3. Filter and sort the data
4. Group players by position to compare market values
5. Make charts to visualize transfer value trends

> 💡 **Key Column:** `market_value_eur` — this is how much a player is worth on the transfer market in Euros.


---
## Step 1 — Import Libraries

We need two libraries:
- **pandas** — for loading and working with the data
- **matplotlib** — for making charts

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print('Libraries imported!')

---
## Step 2 — Load the CSV File

`pd.read_csv()` reads the file and turns it into a DataFrame (a table in Python).

In [ ]:
df = pd.read_csv('fifa_world_cup_2026_player_performance.csv')

print('File loaded!')
print(f'This dataset has {df.shape[0]:,} rows and {df.shape[1]} columns.')

---
## Step 3 — Preview the First 5 Rows

`.head()` shows the first 5 rows so you can see what the data looks like.

In [ ]:
df.head()

---
## Step 4 — See All Column Names

There are a lot of columns in this dataset. Let's list them all.

In [ ]:
print(f'Total columns: {len(df.columns)}')
print()
for col in df.columns:
    print(' -', col)

---
## Step 5 — Check Data Types and Missing Values

`.info()` shows data types and which columns have missing values.

In [ ]:
df.info()

In [ ]:
print('Missing values per column (only showing columns that have some):')
missing = df.isnull().sum()
print(missing[missing > 0])

---
## Step 6 — Basic Statistics

`.describe()` shows the min, max, average, and percentiles for every number column.

In [ ]:
# Only show the most relevant columns
cols = ['age', 'market_value_eur', 'goals', 'assists', 'player_rating', 'minutes_played']
df[cols].describe().round(2)

---
## Step 7 — Look at Unique Positions

`.unique()` shows every different value in a column — like a list of all positions.

In [ ]:
print('Positions in the dataset:', df['position'].unique())
print()
print('How many rows per position:')
print(df['position'].value_counts())

---
## Step 8 — Focus on Key Columns

This dataset has 70+ columns. For our analysis we only need a few.
Let's create a smaller DataFrame with just the columns we care about.

In [ ]:
# Select only the columns we need
cols_we_need = [
    'player_name', 'age', 'nationality', 'position', 'club_name',
    'market_value_eur', 'goals', 'assists', 'player_rating', 'minutes_played'
]

df_clean = df[cols_we_need].copy()

# Convert market value from Euros to Millions (easier to read)
df_clean['market_value_m'] = (df_clean['market_value_eur'] / 1_000_000).round(1)

print('Cleaned DataFrame shape:', df_clean.shape)
df_clean.head()

---
## Step 9 — Aggregate by Player

Each player appears multiple times (once per match). Let's combine rows so each player appears once with their totals.

In [ ]:
# Group by player and sum goals/assists, average rating and market value
player_summary = df_clean.groupby('player_name').agg(
    nationality   = ('nationality',      'first'),
    position      = ('position',         'first'),
    club          = ('club_name',        'first'),
    age           = ('age',              'first'),
    market_value  = ('market_value_m',   'first'),
    total_goals   = ('goals',            'sum'),
    total_assists = ('assists',          'sum'),
    avg_rating    = ('player_rating',    'mean'),
    total_minutes = ('minutes_played',   'sum'),
).reset_index()

player_summary['avg_rating'] = player_summary['avg_rating'].round(1)

print(f'Unique players: {len(player_summary)}')
player_summary.head(10)

---
## Step 10 — Top 10 Players by Market Value

In [ ]:
top10 = player_summary.sort_values('market_value', ascending=False).head(10)

top10[['player_name', 'nationality', 'position', 'club', 'total_goals', 'market_value']]

---
## Step 11 — Average Market Value by Position

In [ ]:
avg_by_pos = player_summary.groupby('position')['market_value'].mean().sort_values(ascending=False)

print('Average Market Value (€M) by Position:')
print(avg_by_pos.round(1))

---
## Step 12 — Chart: Market Value by Position

In [ ]:
colors = ['#1a3a6b', '#2e86c1', '#5dade2', '#a9cce3']

avg_by_pos.plot(kind='bar', color=colors, edgecolor='white', figsize=(8, 5))

plt.title('Average Market Value by Position (€M)', fontsize=14)
plt.xlabel('Position')
plt.ylabel('Market Value (€M)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## Step 13 — Chart: Goals vs. Market Value

Do players who score more goals have higher market values?  
A scatter plot shows us the relationship.

In [ ]:
# Use only top 100 players by value so the chart isn't too crowded
top100 = player_summary.sort_values('market_value', ascending=False).head(100)

pos_colors = {'Forward': '#e74c3c', 'Midfielder': '#1a3a6b',
              'Defender': '#27ae60', 'Goalkeeper': '#f39c12'}

plt.figure(figsize=(9, 6))

for pos, color in pos_colors.items():
    grp = top100[top100['position'] == pos]
    plt.scatter(grp['total_goals'], grp['market_value'],
                label=pos, color=color, alpha=0.75, s=80)

plt.title('Goals vs. Market Value — Top 100 Players by Value', fontsize=13)
plt.xlabel('Total Goals in Tournament')
plt.ylabel('Market Value (€M)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 14 — Filter: High-Value Players Who Scored 10+ Goals

In [ ]:
# Players with 10+ goals AND market value over €50M
elite = player_summary[
    (player_summary['total_goals'] >= 10) &
    (player_summary['market_value'] >= 50)
].sort_values('market_value', ascending=False)

print(f'Elite scorers (10+ goals, €50M+ value): {len(elite)} players')
elite[['player_name', 'nationality', 'position', 'total_goals', 'total_assists', 'market_value']]

---
## ✅ Cheat Sheet

| Code | What it does |
|------|--------------|
| `pd.read_csv('file.csv')` | Load CSV into DataFrame |
| `df.head()` | First 5 rows |
| `df.shape` | (rows, columns) |
| `df.columns` | All column names |
| `df.info()` | Summary + data types |
| `df.describe()` | Stats (min/max/avg) |
| `df.isnull().sum()` | Missing value count |
| `df['col'].unique()` | Unique values |
| `df['col'].value_counts()` | Count of each value |
| `df[cols].copy()` | Select specific columns |
| `df.groupby('col').agg(...)` | Aggregate by group |
| `df[condition]` | Filter rows |
| `df.sort_values('col')` | Sort rows |

---
**Next:** Open `02_superbowl_ad_revenue.ipynb` to analyze Super Bowl ad pricing!